# 2015–16 CRDC data cleaning

This notebook cleans the 2015–16 Civil Rights Data Collection (CRDC) school and LEA files using the sequence in `data_audit.md`.

2015–16 is usable only after repairing Excel-corrupted IDs. Negative reserved codes are missingness, not counts — they are recoded to NA and never filled with 0.

Cleaning sequence:
1. Read CSVs as strings (UTF-8, fall back to cp1252); drop empty rows.
2. Pad `LEAID` / `SCHID` and **rebuild** `COMBOKEY` (do not use the stored column).
3. Left-join LEA characteristics on padded `LEAID`.
4. Recode reserved codes to NA; keep the original code; build seclusion instance totals, enrollment, and rates.
5. Flag implausible FTE values (including 999 placeholders).
6. Flag JJ / special-education / small-enrollment schools.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from clean_crdc_2015_16 import (
    COUNT_FTE_COLS,
    FTE_ALL_COLS,
    FTE_PLACEHOLDER_COLS,
    LEA_COLS,
    LEA_PATH,
    OUT_DIR,
    RESERVED_GROUPS,
    SCH_COLS,
    SCH_PATH,
    SECL_INSTANCE_COLS,
    pad_id,
    print_audit_checks,
    read_crdc_csv,
    recode_reserved,
    yes_no_flag,
)

## 1. Read priority columns as strings

2015–16 files are `cp1252`. The reader tries UTF-8 first and falls back. Every value is a string so IDs keep leading zeros.

In [ ]:
sch = read_crdc_csv(SCH_PATH, SCH_COLS)
lea = read_crdc_csv(LEA_PATH, LEA_COLS)
sch.shape, lea.shape

## 2. Rebuild `COMBOKEY`

The stored `COMBOKEY` dropped leading zeros and used scientific notation (29,374 distinct values). Rebuild it as padded `LEAID` (7) + `SCHID` (5).

In [ ]:
sch["COMBOKEY_stored"] = sch["COMBOKEY"]
sch["LEAID"] = pad_id(sch["LEAID"], 7)
sch["SCHID"] = pad_id(sch["SCHID"], 5)
sch["COMBOKEY"] = sch["LEAID"] + sch["SCHID"]
lea["LEAID"] = pad_id(lea["LEAID"], 7)

print("Stored COMBOKEY unique:", sch["COMBOKEY_stored"].nunique())
print("Rebuilt COMBOKEY unique:", sch["COMBOKEY"].nunique(), "of", len(sch), "rows")
print("Duplicate rebuilt keys:", sch["COMBOKEY"].duplicated().sum())
print("Schools with no LEA match:", (~sch["LEAID"].isin(lea["LEAID"])).sum())

## 3. Join school file to LEA characteristics

2015–16 is one wide school file, so there is no school-module inner join. LEA fields (`LEA_ENR`, `LEA_SCHOOLS`) are left-joined on `LEAID`.

In [ ]:
df = sch.merge(lea, on="LEAID", how="left", suffixes=("", "_lea"))
df.shape

## 4. Recode reserved codes; build outcomes

Values `< 0` are reserved codes (`-9` skip, `-5` action plan, `-2` small cell, etc.). Recode them to NA and keep the original code. Zeros stay zeros.

Seclusion instances are summed only when WODIS, IDEA, and 504 are all valid. The rate is `100 * instances / (TOT_ENR_M + TOT_ENR_F)` and is NA when enrollment is missing or 0.

In [ ]:
df = recode_reserved(df, COUNT_FTE_COLS)
extra = {}

secl_valid = df[SECL_INSTANCE_COLS].notna().all(axis=1)
extra["seclusion_instances"] = np.where(
    secl_valid, df[SECL_INSTANCE_COLS].sum(axis=1), np.nan
)
wodis_code = df["SCH_RSINSTANCES_SECL_WODIS_code"]
extra["seclusion_reason_code"] = wodis_code.fillna(
    df["SCH_RSINSTANCES_SECL_IDEA_code"]
).fillna(df["SCH_RSINSTANCES_SECL_504_code"])
extra["seclusion_reason"] = extra["seclusion_reason_code"].map(RESERVED_GROUPS)

enr_valid = df["TOT_ENR_M"].notna() & df["TOT_ENR_F"].notna()
extra["enrollment"] = np.where(enr_valid, df["TOT_ENR_M"] + df["TOT_ENR_F"], np.nan)
extra["seclusion_rate_per_100"] = np.where(
    (extra["enrollment"] > 0) & secl_valid,
    100.0 * extra["seclusion_instances"] / extra["enrollment"],
    np.nan,
)
extra["flag_missing_seclusion"] = ~secl_valid
extra["flag_missing_enrollment"] = ~enr_valid
extra["flag_zero_enrollment"] = extra["enrollment"] == 0
extra["flag_rate_undefined"] = (
    extra["flag_missing_seclusion"]
    | extra["flag_missing_enrollment"]
    | extra["flag_zero_enrollment"]
)

print("All three seclusion fields valid:", int(secl_valid.sum()))
print("WODIS -9:", int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -9).sum()))
print("WODIS -5:", int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -5).sum()))
print("WODIS > 0:", int((df["SCH_RSINSTANCES_SECL_WODIS"] > 0).sum()))
print("IDEA > 0:", int((df["SCH_RSINSTANCES_SECL_IDEA"] > 0).sum()))
print("TOT_ENR_M usable:", int(df["TOT_ENR_M"].notna().sum()))
print("TOT_ENR_M = 0:", int((df["TOT_ENR_M"] == 0).sum()))
print("M + F = 0:", int((extra["enrollment"] == 0).sum()))

## 5–6. FTE plausibility and school-type flags

Counselor / nurse / psychologist / social-worker FTE of 999 is treated as a placeholder and set to NA. Other FTE fields are flagged when FTE exceeds enrollment, or when FTE per 100 students is above 20 (50 for teachers). There is no virtual-school indicator in 2015–16.

In [ ]:
enrollment = extra["enrollment"]

for prefix, m_col, f_col in [
    ("idea", "SCH_ENR_IDEA_M", "SCH_ENR_IDEA_F"),
    ("sec504", "SCH_ENR_504_M", "SCH_ENR_504_F"),
    ("lep", "SCH_ENR_LEP_M", "SCH_ENR_LEP_F"),
    ("secl_students_nonidea", "TOT_RS_NONIDEA_SECL_M", "TOT_RS_NONIDEA_SECL_F"),
    ("secl_students_idea", "TOT_RS_IDEA_SECL_M", "TOT_RS_IDEA_SECL_F"),
]:
    both = df[m_col].notna() & df[f_col].notna()
    extra[f"{prefix}_n"] = np.where(both, df[m_col] + df[f_col], np.nan)
    if prefix in {"idea", "sec504", "lep"}:
        extra[f"{prefix}_share"] = np.where(
            enrollment > 0, extra[f"{prefix}_n"] / enrollment, np.nan
        )

extra["flag_fte_placeholder_999"] = False
fte_updates = {}
for col in FTE_PLACEHOLDER_COLS:
    is_999 = df[col].eq(999)
    extra[f"{col}_placeholder_999"] = is_999
    extra["flag_fte_placeholder_999"] = extra["flag_fte_placeholder_999"] | is_999
    updated = df[col].copy()
    updated.loc[is_999] = np.nan
    fte_updates[col] = updated
df[list(fte_updates)] = pd.DataFrame(fte_updates, index=df.index)

extra["flag_fte_outlier"] = False
teacher_cols = {"SCH_FTETEACH_TOT", "SCH_FTETEACH_CERT"}
for col in FTE_ALL_COLS:
    per_100 = np.where(enrollment > 0, 100.0 * df[col] / enrollment, np.nan)
    extra[f"{col}_per_100"] = per_100
    exceeds_enrollment = df[col].notna() & pd.notna(enrollment) & (df[col] > enrollment)
    cap = 50 if col in teacher_cols else 20
    outlier = exceeds_enrollment | (df[col].notna() & (per_100 > cap))
    extra[f"{col}_outlier"] = outlier
    extra["flag_fte_outlier"] = extra["flag_fte_outlier"] | outlier.fillna(False)

extra["flag_jj"] = yes_no_flag(df["JJ"])
extra["flag_sped"] = yes_no_flag(df["SCH_STATUS_SPED"])
extra["flag_charter"] = yes_no_flag(df["SCH_STATUS_CHARTER"])
extra["flag_alt"] = yes_no_flag(df["SCH_STATUS_ALT"])
extra["flag_magnet"] = yes_no_flag(df["SCH_STATUS_MAGNET"])
extra["flag_small_enrollment"] = pd.notna(enrollment) & (enrollment < 50)
extra["year"] = "2015-16"

df = pd.concat([df, pd.DataFrame(extra, index=df.index)], axis=1)
print("JJ schools:", int(df["flag_jj"].sum()))
print("SPED schools:", int(df["flag_sped"].sum()))
print("Enrollment < 50:", int(df["flag_small_enrollment"].sum()))
print("FTE 999 placeholder schools:", int(df["flag_fte_placeholder_999"].sum()))
print("FTE outlier schools:", int(df["flag_fte_outlier"].sum()))

## Audit checks and export

Compare cleaned totals to the 2015–16 counts in the data audit, then write the analysis-ready table.

In [ ]:
report = {
    "n_schools": len(df),
    "unique_combokey": df["COMBOKEY"].nunique(),
    "stored_combokey_unique": df["COMBOKEY_stored"].nunique(),
    "unmatched_lea": int(df["LEA_ENR"].isna().sum()),
    "jj_yes": int(df["flag_jj"].sum()),
    "seclusion_all_three_valid": int((~df["flag_missing_seclusion"]).sum()),
    "wodis_neg9": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -9).sum()),
    "wodis_neg5": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -5).sum()),
    "wodis_positive": int((df["SCH_RSINSTANCES_SECL_WODIS"] > 0).sum()),
    "idea_positive": int((df["SCH_RSINSTANCES_SECL_IDEA"] > 0).sum()),
    "enr_m_usable": int(df["TOT_ENR_M"].notna().sum()),
    "enr_f_usable": int(df["TOT_ENR_F"].notna().sum()),
    "enr_m_zero": int((df["TOT_ENR_M"] == 0).sum()),
    "enr_sum_zero": int((df["enrollment"] == 0).sum()),
    "fte_placeholder_rows": int(df["flag_fte_placeholder_999"].sum()),
    "fte_outlier_rows": int(df["flag_fte_outlier"].sum()),
}
print_audit_checks(report)

preview_cols = [
    "year", "COMBOKEY", "LEAID", "SCHID", "LEA_STATE", "SCH_NAME",
    "seclusion_instances", "enrollment", "seclusion_rate_per_100",
    "seclusion_reason", "flag_jj", "flag_sped", "flag_small_enrollment",
    "flag_fte_placeholder_999", "flag_fte_outlier",
]
df[preview_cols].head()

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
out_csv = OUT_DIR / "crdc_2015_16_analysis_ready.csv"
df.to_csv(out_csv, index=False)
print(f"Wrote {out_csv} ({len(df):,} rows, {df.shape[1]} columns)")